# Track-A v1.2 — MASTER K2 (v11)

Exact qualified science: `56023042e57758591df9babb3438f191dbe10312`  
Immutable operator runtime: `0963f81ec54b4ac121097856741616ac04530797`  
Runtime branch: `ops-tracka-kaggle-master-runtime-v11-0963f81`

Use Kaggle **T4 x2**, **Internet ON**, and **Save Version → Save & Run All / Batch**. Do not edit scientific settings.
Required Kaggle secrets: `KAGGLE_USERNAME`, `KAGGLE_KEY`, `CROPCOP_GITHUB_TOKEN`, and `CROPCOP_EXPECTED_KAGGLE_USERNAME`. The expected-username secret must equal this Kaggle account.
Recommended: launch only after K1 publishes the exact v11 G1A READY handoff and this account has `Can view` access to the private G1A dataset. If launched early, v11 exits cleanly with `rc=2` before expensive stack installation instead of waiting.


In [ ]:
from pathlib import Path
import os

os.environ['PYTHONUNBUFFERED'] = '1'
V1 = Path('/kaggle/input/datasets/ranamuhammadahmed6/cropcop-finalized-v8-11-2026-1/CropCop_Final_v1')
if V1.is_dir():
    preferred = {
        'CROPCOP_MANIFEST': V1 / 'audit' / 'final_manifest.csv',
        'CROPCOP_CLASS_MAP': V1 / 'audit' / 'class_to_idx.json',
        'CROPCOP_IMAGE_ROOT': V1 / 'dataset',
    }
    for name, path in preferred.items():
        if path.exists():
            os.environ.setdefault(name, str(path))

expected = os.environ.get('CROPCOP_EXPECTED_KAGGLE_USERNAME', '').strip()
if not expected:
    try:
        from kaggle_secrets import UserSecretsClient
        expected = UserSecretsClient().get_secret('CROPCOP_EXPECTED_KAGGLE_USERNAME').strip()
    except Exception as exc:
        raise RuntimeError('Add Kaggle secret CROPCOP_EXPECTED_KAGGLE_USERNAME with this account username before running v11.') from exc
if not expected:
    raise RuntimeError('CROPCOP_EXPECTED_KAGGLE_USERNAME is empty.')
os.environ['CROPCOP_EXPECTED_KAGGLE_USERNAME'] = expected
print('Expected account binding loaded for K2.', flush=True)


In [ ]:
from pathlib import Path
import shutil, subprocess, sys

OPS_RUNTIME_SHA = '0963f81ec54b4ac121097856741616ac04530797'
OPS_RUNTIME_BRANCH = 'ops-tracka-kaggle-master-runtime-v11-0963f81'
OPS_ROOT = Path('/kaggle/working/cropcop-tracka-master-runtime-v11')
if OPS_ROOT.exists():
    shutil.rmtree(OPS_ROOT)
subprocess.run([
    'git', 'clone', '--quiet', '--depth', '1', '--branch', OPS_RUNTIME_BRANCH,
    'https://github.com/rana-m-ahmed/ResearchWork-CropCop.git', str(OPS_ROOT)
], check=True)
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=OPS_ROOT, text=True).strip()
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=OPS_ROOT, text=True).strip()
if head != OPS_RUNTIME_SHA:
    raise RuntimeError(f'operator runtime SHA mismatch: expected {OPS_RUNTIME_SHA}, got {head}')
if dirty:
    raise RuntimeError(f'operator runtime checkout is dirty: {dirty}')
print('Pinned Track-A v11 runtime:', head, flush=True)


In [ ]:
guard = OPS_ROOT / 'journal_extension/kaggle/tracka_v12_ops/master_launch_guard_v11.py'
env = dict(os.environ)
env['PYTHONUNBUFFERED'] = '1'
proc = subprocess.Popen(
    [sys.executable, '-u', str(guard), 'K2'],
    cwd=guard.parent,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert proc.stdout is not None
for line in proc.stdout:
    print(line, end='', flush=True)
rc = proc.wait()
if rc == 0:
    print('K2 v11 master: TERMINAL PASS for this account queue.', flush=True)
elif rc == 2:
    print('K2 v11 master: controlled continuation. End this Batch and rerun this SAME v11 notebook after the stated prerequisite is ready.', flush=True)
else:
    raise RuntimeError('K2 v11 master requires investigation; rc=' + str(rc))


v11 reliability contract: a `MASTER_HEARTBEAT_V11` line is emitted every ~60 seconds during long driver stages. `rc=2` means safely published work or an unmet prerequisite; rerun this same notebook later in a fresh Batch session. Any other non-zero return code is fail-closed and requires investigation.
